# Импортирование необходимых модулей

In [2]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import numpy.ma as ma
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Rectangle
import seaborn as sns
from scipy import stats
from scipy.ndimage import binary_dilation

from skimage.filters import threshold_otsu

from datetime import datetime
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

import binascii
import io
from io import BytesIO
from struct import *

import base64

from MDTdeclaration import *
from MDTfile import *

In [4]:
matplotlib.rcParams['axes.unicode_minus'] = False

def create_afm_report_docx(input_filename, operator, outdir = './'):
    """
    Создает Word документ с отчетом по AFM
    
    Parameters:
    -----------
    mdt_flie : MDTfile
        загруженный файл
    
    operator: str
        имя того кто составлял логбук
    
    output_filename : str
        Имя выходного файла Word
    
    Returns:
    --------
    str : путь к созданному файлу
    """
    #print(images_data)
    
    sample, report_number = sample_name(input_filename)
    report_name = f'{outdir}{sample} AFM {report_number} jupyter.docx'
    mdt_file = MDTFile()
    mdt_file.load_mdt_file(input_filename)
    mdt_file_metadata = {
        'filename': input_filename,
        'frames': []
    }
    
    
    # Создаем документ
    doc = Document()
    
    # Настраиваем шрифт для поддержки русского языка
    style = doc.styles['Normal']
    style.font.name = 'Times New Roman'
    style._element.rPr.rFonts.set(qn('w:eastAsia'), 'Times New Roman')
    style.font.size = Pt(11)
    
    
    
    # Добавляем дату
    date_para = doc.add_paragraph()
    date_run = date_para.add_run(f'#date {datetime.now().strftime("%Y-%m-%d")}')
    #date_run.bold = True
    date_para.alignment = WD_ALIGN_PARAGRAPH.LEFT
    doc.add_paragraph(f"#operator {operator}")
    doc.add_paragraph(f"#action AFM")
    doc.add_paragraph(f"#sample {sample}")
    doc.add_paragraph()  # Пустая строка
    doc.add_paragraph(f'Имя файла: {input_filename.split('/')[-1]}')
    
    
    
    for frame in mdt_file:
        if frame.title in ['Text Frame', 'Simple statistics']: doc.add_paragraph(frame.data);continue
        frame = process_frame(frame)
        if frame.title in ['Histogram', '', 'Peak Distribution', '1F:Height Histogram']: continue
        
        doc.add_paragraph(f'Канал: {frame.title}')
        fig1 = create_main_afm_image(frame)
        img_stream1 = save_matplotlib_fig_to_stream(fig1)
        doc.add_picture(img_stream1, width=Inches(6))
        plt.close(fig1)
    
        roughness = {
        'RMS = /<(a - a*)^2>': frame.rms,
        'RMS masked': frame.rms_masked,
        'peak-to-peak': frame.peak_to_peak
        }
        
        doc.add_paragraph(f'Размер скана: {round(frame.xreal, 1)} х {round(frame.yreal, 1)} мкм')
        doc.add_paragraph("Шероховатость:")
        for key, value in roughness.items():
            doc.add_paragraph(f"    {key} = {value:.2f} нм")
    
        doc.add_page_break()
        frame_metadata = {
            'channel': frame.title,
            'size': round(frame.xreal, 1),
            'RMS': frame.rms,
            'RMS masked': frame.rms_otsu,
            'peak_to_peak': frame.peak_to_peak
            }
        mdt_file_metadata['frames'].append(frame_metadata)
    # Сохраняем документ
    print(f"Имя файла отчета: {report_name}")
    doc.save(report_name)
    print(f"Успех")
    
    return None


def save_matplotlib_fig_to_stream(fig, dpi=150):
    """
    Сохраняет matplotlib figure в BytesIO stream
    
    Parameters:
    -----------
    fig : matplotlib.figure.Figure
        Figure для сохранения
    dpi : int
        Разрешение изображения
    
    Returns:
    --------
    BytesIO : поток с изображением
    """
    stream = BytesIO()
    fig.savefig(stream, format='png', dpi=dpi, bbox_inches='tight', facecolor='white')
    stream.seek(0)
    return stream


def create_main_afm_image(frame):
    """
    Создает основное изображение для отчета
    """

    
    fig, [[ax1, ax2], [ax3, ax4]] = plt.subplots(2, 2, figsize = (9, 9))

    
    vmin = np.percentile(frame.data_corrected, 0.5)
    vmax = np.percentile(frame.data_corrected, 99.5)
    
    im = ax1.imshow(frame.data_corrected, 
                    extent = [0, frame.xreal, 0, frame.yreal], 
                    cmap = 'afmhot',
                    vmin = vmin,
                    vmax = vmax
                   )
    plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04, label='Высота, нм')
    ax1.set_xlabel('x, nm')
    ax1.set_ylabel('y, nm')
    

    
    # Гистограмма
    try:
        ax2.hist(frame.data_corrected.flatten(), bins=50, alpha=0.7, 
             color='blue', edgecolor='black')
    except:
        ax2.text(0, 0, 'Не удалось построить гистограмму')
    ax2.set_title('Гистограмма высот')
    ax2.set_xlabel('Высота, нм')
    #ax2.set_ylabel('Частота')
    ax2.grid(True, alpha=0.3)
    
    
    # Добавляем статистику на гистограмму
    stats_text = (f"Min: {frame.data_corrected.min():.3f}\n"
                  f"Max: {frame.data_corrected.max():.3f}\n"
                  f"Mean: {frame.data_corrected.mean():.3f}\n"
                  f"Std: {frame.data_corrected.std():.3f}")
    
    ax2.text(0.02, 0.98, stats_text,
             transform=ax2.transAxes,
             verticalalignment='top',
             fontsize=9,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    try:
        ax3.plot(frame.data_corrected[round(frame.xn/2)], label = f'профиль по х\nстрока {round(frame.xn/2)}')
    except:
        ax3.plot(frame.data_corrected[0], label = f'профиль по х\nстрока 1')
    ax3.legend()
    try:
        ax4.plot(frame.data_corrected.T[round(frame.yn/2)], label = f'профиль по y\nстолбец {round(frame.yn/2)}')
    except:
        ax4.plot(frame.data_corrected.T[0], label = f'профиль по y\nстолбец 1')
    ax4.legend()
    ax3.set_xlabel('x, nm')
    ax3.set_ylabel('z, nm')
    ax4.set_xlabel('y, nm')
    ax4.set_ylabel('z, nm')
    
    plt.tight_layout()
    return fig

In [5]:

def fit_lines_basic(image_data, fit_type='linear', subtract=True):
    """
    Выравнивание AFM изображения построчно (line by line leveling).
    
    Параметры:
    ----------
    image_data : numpy.ndarray
        2D массив с AFM данными (высоты)
    fit_type : str
        Тип аппроксимации для каждой строки:
        - 'mean': вычитание среднего значения строки
        - 'linear': линейная аппроксимация (y = ax + b)
        - 'quadratic': квадратичная аппроксимация (y = ax² + bx + c)
        - 'plane': плоскость для каждой строки (вычитание наклона)
    subtract : bool
        Если True - возвращает выровненные данные,
        Если False - возвращает массив поправок (фитов) для каждой строки
    
    Возвращает:
    -----------
    numpy.ndarray
        Выровненное изображение или массив поправок в зависимости от параметра subtract
    """
    
    if not isinstance(image_data, np.ndarray) or image_data.ndim != 2:
        raise ValueError("image_data должен быть 2D numpy массивом")
    
    h, w = image_data.shape
    x = np.arange(w)  # координаты пикселей вдоль строки
    
    # Создаем массив для выровненного изображения или поправок
    if subtract:
        leveled_image = np.zeros_like(image_data)
    else:
        fits = np.zeros_like(image_data)
    
    # Обрабатываем каждую строку
    for i in range(h):
        line = image_data[i, :]
        
        if fit_type == 'mean':
            # Просто вычитаем среднее значение строки
            fit_value = np.mean(line)
            fit_line = np.full(w, fit_value)
            
        elif fit_type == 'linear':
            # Линейная регрессия: y = ax + b
            slope, intercept, _, _, _ = stats.linregress(x, line)
            fit_line = slope * x + intercept
            
        elif fit_type == 'quadratic':
            # Квадратичная аппроксимация: y = ax² + bx + c
            coeffs = np.polyfit(x, line, 2)
            fit_line = coeffs[0] * x**2 + coeffs[1] * x + coeffs[2]
            
        elif fit_type == 'plane':
            # Аппроксимация плоскостью для строки
            # Для каждой строки рассматриваем как 1D плоскость
            slope, intercept, _, _, _ = stats.linregress(x, line)
            fit_line = slope * x + intercept
            
        else:
            raise ValueError(f"Неизвестный тип fit_type: {fit_type}. "
                           f"Доступные варианты: 'mean', 'linear', 'quadratic', 'plane'")
        
        if subtract:
            # Вычитаем фит из оригинальных данных
            leveled_image[i, :] = line - fit_line
        else:
            # Сохраняем сам фит
            fits[i, :] = fit_line
    
    if subtract:
        return leveled_image
    else:
        return fits
    
    
def fit_lines(image_data, fit_type='quadratic', order=1, percentile=95, mask_dilation=2):
    """
    Быстрое выравнивание AFM изображения с автоматическим удалением островков.
    
    Параметры:
    ----------
    image_data : numpy.ndarray
        2D массив с AFM данными
    fit_type : str
        Тип аппроксимации для финального прохода:
        - 'poly': полином указанного порядка (использует параметр order)
        - 'linear': линейная аппроксимация (order=1)
        - 'quadratic': квадратичная аппроксимация (order=2)
        - 'cubic': кубическая аппроксимация (order=3)
        - 'mean': вычитание среднего
    order : int
        Порядок полинома для fit_type='poly' (по умолчанию 1)
    percentile : float
        Процентиль для определения островков (по умолчанию 95)
    mask_dilation : int
        Радиус расширения маски (чтобы захватить края островков)
    
    Возвращает:
    -----------
    numpy.ndarray
        Выровненное изображение
    
    
    h, w = image_data.shape
    
    # Шаг 1: грубое выравнивание глобальной плоскостью
    #print("Шаг 1/3: Глобальное выравнивание для построения маски...")
    X, Y = np.meshgrid(np.arange(w), np.arange(h))
    
    # Вычитаем глобальную плоскость (всегда линейная для первого прохода)
    X_flat = X.flatten()
    Y_flat = Y.flatten()
    Z_flat = image_data.flatten()
    A = np.column_stack([X_flat, Y_flat, np.ones_like(X_flat)])
    coeffs, _, _, _ = np.linalg.lstsq(A, Z_flat, rcond=None)
    plane = coeffs[0] * X + coeffs[1] * Y + coeffs[2]
    temp_leveled = image_data - plane
"""
    h, w = image_data.shape
    temp_leveled = fit_lines_basic(image_data)
    #plt.imshow(temp_leveled)
    #plt.show()
    # Шаг 2: строим маску на временном изображении
    #print("Шаг 2/3: Построение маски островков...")
    threshold = np.percentile(temp_leveled, percentile)
    mask = temp_leveled > threshold
    #plt.imshow(mask)
    #plt.show()
    
    # Расширяем маску для захвата краев
    if mask_dilation > 0:
        struct = np.ones((mask_dilation, mask_dilation))
        mask = binary_dilation(mask, structure=struct)
    
    masked_percent = np.sum(mask) * 100 / mask.size
    #print(f"  Маскировано {masked_percent:.1f}% пикселей")
    
    # Шаг 3: финальное построчное выравнивание с маской
    #print(f"Шаг 3/3: Финальное выравнивание (fit_type={fit_type}, order={order})...")
    leveled = np.zeros_like(image_data)
    x = np.arange(w)
    
    # Определяем порядок полинома
    if fit_type == 'linear':
        poly_order = 1
    elif fit_type == 'quadratic':
        poly_order = 2
    elif fit_type == 'cubic':
        poly_order = 3
    elif fit_type == 'poly':
        poly_order = order
    elif fit_type == 'mean':
        poly_order = 0
    else:
        raise ValueError(f"Неизвестный fit_type: {fit_type}. Доступны: 'poly', 'linear', 'quadratic', 'cubic', 'mean'")
    
    for i in range(h):
        line = image_data[i, :]
        line_mask = mask[i, :]
        
        # Если вся строка замаскирована, используем все точки
        if np.all(line_mask):
            x_clean = x
            y_clean = line
        else:
            x_clean = x[~line_mask]
            y_clean = line[~line_mask]
        
        # Проверяем, достаточно ли точек для аппроксимации
        min_points = poly_order + 1
        if len(x_clean) < min_points:
            # Недостаточно точек - используем все и уменьшаем порядок
            x_clean = x
            y_clean = line
            actual_order = min(1, len(x_clean) - 1)
            if actual_order < 0:
                actual_order = 0
        else:
            actual_order = poly_order
        
        # Аппроксимация полиномом
        if actual_order == 0:
            # Среднее значение
            fit_line = np.full(w, np.mean(y_clean))
        else:
            # Полином указанного порядка
            coeffs = np.polyfit(x_clean, y_clean, actual_order)
            fit_line = np.polyval(coeffs, x)
        
        leveled[i, :] = line - fit_line
    
    return leveled


In [6]:

def calculate_basic_statistics(image_data):
    """
    Рассчитывает базовую статистику изображения
    """
    '''print(f"\nСтатистика канала '{channel_name}':")
    
    # Основные статистики
    stats = {
        'Минимум': np.min(image_data),
        'Максимум': np.max(image_data),
        'Среднее': np.mean(image_data),
        'Медиана': np.median(image_data),
        'Стандартное отклонение': np.std(image_data),
    }
    
    for key, value in stats.items():
        print(f"  {key}: {value:.6f}")'''
    
    # Шероховатость
    flattened = image_data.flatten()
    mean_val = np.mean(flattened)
    
    roughness = {
        'Ra (средняя шероховатость) = <|a - a*|>': np.mean(np.abs(flattened - mean_val)),
        'RMS = /<(a - a*)^2>': np.sqrt(np.mean((flattened - mean_val)**2)),
        'peak-to-peak': np.max(flattened) - np.min(flattened)
    }
    
    #print("\n  Шероховатость:")
    for key, value in roughness.items():
        print(f"    {key} = {value:.6f}")
    
    return {**roughness}

In [7]:
def calculate_rms_otsu(data):
    """
    Автоматическое определение порога методом Оцу
    """
    
    # Нормализуем данные для метода Оцу
    data_norm = (data - np.min(data)) / (np.max(data) - np.min(data))
    data_norm = (data_norm * 255).astype(np.uint8)
    
    # Находим порог методом Оцу
    try:
        threshold_norm = threshold_otsu(data_norm) / 255.0
        threshold = np.min(data) + threshold_norm * (np.max(data) - np.min(data))
    except:
        # Fallback на процентиль
        threshold = np.percentile(data, 95)
    
    # Маскируем точки выше порога
    mask_data = data[data< threshold]
    rms = np.sqrt(np.mean((mask_data - np.mean(mask_data))**2))
    
    masked_data = np.where(data > threshold, np.nan, data)
    
    #mean_masked = np.mean(masked_data)
    #rms = np.sqrt(np.mean((masked_data - mean_masked)**2))
    
    return rms, masked_data

In [8]:
def analyze_roughness_with_plots(frame, method='gaussian', n_sigma=2):
    """
    Анализ шероховатости с визуализацией порога
    """
    data = frame.data_corrected.flatten()
    
    if method == 'gaussian':
        result = calculate_roughness_gaussian(frame, n_sigma=n_sigma)
        threshold = result['threshold']
    elif method == 'otsu':
        result = calculate_roughness_otsu(frame)
        threshold = result['threshold']
    elif method == 'percentile':
        result = calculate_roughness_threshold(frame, threshold_percentile=95)
        threshold = result['threshold']
    elif method == 'gmm':
        result = calculate_roughness_gmm(frame)
        threshold = result['threshold']
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Визуализация
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # 1. Исходное изображение
    ax1 = axes[0, 0]
    im = ax1.imshow(frame.data_corrected, cmap='afmhot')
    ax1.set_title('Исходное изображение')
    plt.colorbar(im, ax=ax1, label='Высота, нм')
    
    # 2. Маскированное изображение
    ax2 = axes[0, 1]
    masked_data = frame.data_corrected.copy()
    masked_data[masked_data > threshold] = np.nan
    im2 = ax2.imshow(masked_data, cmap='viridis')
    ax2.set_title(f'Маскированное (порог = {threshold:.1f} нм)')
    plt.colorbar(im2, ax=ax2, label='Высота, нм')
    
    # 3. Гистограмма с порогом
    ax3 = axes[1, 0]
    hist, bins, _ = ax3.hist(data, bins=100, alpha=0.7, color='steelblue', 
                              edgecolor='black', label='Все данные')
    ax3.axvline(threshold, color='red', linewidth=2, label=f'Порог = {threshold:.1f} нм')
    ax3.set_xlabel('Высота, нм')
    ax3.set_ylabel('Частота')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Статистика
    ax4 = axes[1, 1]
    ax4.axis('off')
    stats_text = f"""
    Результаты расчета шероховатости (метод: {method})
    
    RMS:           {result['RMS']:.2f} нм
    Peak-to-peak:  {result['Peak-to-peak']:.2f} нм
    Порог:         {result['threshold']:.2f} нм
    
    Удалено точек: {result['points_removed']})
    """
    
    if 'mean' in result:
        stats_text += f"\n    Среднее:       {result['mean']:.2f} нм"
        stats_text += f"\n    Сигма:         {result['sigma']:.2f} нм"
    
    if 'substrate_mean' in result:
        stats_text += f"\n\n    Подложка:      {result['substrate_mean']:.2f} ± {result['substrate_std']:.2f} нм"
        stats_text += f"\n    Островки:      {result['islands_mean']:.2f} нм"
    
    ax4.text(0.1, 0.5, stats_text, fontsize=11, verticalalignment='center',
             fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    return result

# Пример использования
#result = analyze_roughness_with_plots(frame, method='gaussian', n_sigma=4)
#print(f"RMS = {result['RMS']:.2f} нм")

In [9]:
def sample_name(filename):
    #print(filename)
    file = filename.split('/')[-1]
    #print(file)
    #print(file[-6])
    #print(file.startswith('a') and file[-6] == '0')
    if file.startswith('a') and file[-6] == '0': 
        try:
            sample = file.split('-')[-2][1:]
            report_number = file[-6:-4]
        except:
            sample = file[:-4]
            report_number = '01'            
    else:
        sample = file[:-4]
        report_number = '01'
    return sample, report_number
    
def mdt_processing(filename, operator = 'jupyter'):
    '''
    Быстрая обработка файлов, сейчас не используется
    '''
    print(filename)
    mdt_file = MDTFile()

    mdt_file.load_mdt_file(filename)
    for frame in mdt_file:
        if frame.title in ['Text Frame', 'Histogram', '', 'Peak Distribution', '1F:Height Histogram', 'Simple statistics']: 
            #print(frame.title)
            continue
        #print(frame.dimensions_unit)
        frame.data_corrected = fit_lines(frame.data, fit_type='linear', subtract=True)
        frame.data_corrected = frame.data_corrected - np.min(frame.data_corrected)
        frame.data_corrected = frame.data_corrected * 10**3
    sample, report_number = sample_name(filename)
    report_name = f'./new_reports/{sample} AFM {report_number} jupyter.docx'
    input_filename = filename.split('/')[-1]
    create_afm_report_docx(mdt_file, operator, input_filename, report_name)
    del mdt_file
    return None

def process_frame(frame):
    '''
    Предобработка фреймов: вычисение шероховатости, построчная коррекция
    '''
    if frame.title in ['Text Frame', 'Histogram', '', 
                       'Peak Distribution', '1F:Height Histogram', 'Simple statistics']: 
        return frame
        #print(frame.title)
        
        #print(frame.dimensions_unit)
    frame.data_corrected = fit_lines(frame.data, fit_type='quadratic')
    #frame.data_corrected = fit_lines_otzu()
    frame.data_corrected = frame.data_corrected - np.min(frame.data_corrected)
    frame.data_corrected = frame.data_corrected - np.min(frame.data_corrected)
    frame.data_corrected = frame.data_corrected * 10**3
    #mask = (frame.data_corrected > np.percentile(frame.data_corrected, 99.5))
    mask = (frame.data_corrected > np.min(frame.data_corrected) + 5)
    frame.data_masked = ma.masked_where(mask, frame.data_corrected)
    flattened = frame.data_corrected.flatten()
    mean_val = np.mean(flattened)
    frame.rms = round(np.sqrt(np.mean((flattened - mean_val)**2)), 2)
    frame.peak_to_peak = round(np.max(flattened) - np.min(flattened), 2)
    flattened = frame.data_masked.flatten()
    mean_val = np.mean(flattened)
    frame.rms_masked = round(np.sqrt(np.mean((flattened - mean_val)**2)), 2)
    frame.rms_otsu, frame.masked_otsu = calculate_rms_otsu(frame.data_corrected)
    return frame

In [10]:
def fit_lines_otzu(image_data, fit_type='quadratic', order=1, percentile=95, mask_dilation=2):
    """
    Быстрое выравнивание AFM изображения с автоматическим удалением островков с использованием порога по Оцу.
    
    Параметры:
    ----------
    image_data : numpy.ndarray
        2D массив с AFM данными
    fit_type : str
        Тип аппроксимации для финального прохода:
        - 'poly': полином указанного порядка (использует параметр order)
        - 'linear': линейная аппроксимация (order=1)
        - 'quadratic': квадратичная аппроксимация (order=2)
        - 'cubic': кубическая аппроксимация (order=3)
        - 'mean': вычитание среднего
    order : int
        Порядок полинома для fit_type='poly' (по умолчанию 1)
    percentile : float
        Процентиль для определения островков (по умолчанию 95)
    mask_dilation : int
        Радиус расширения маски (чтобы захватить края островков)
    
    Возвращает:
    -----------
    numpy.ndarray
        Выровненное изображение
    
    
    h, w = image_data.shape
    
    # Шаг 1: грубое выравнивание глобальной плоскостью
    #print("Шаг 1/3: Глобальное выравнивание для построения маски...")
    X, Y = np.meshgrid(np.arange(w), np.arange(h))
    
    # Вычитаем глобальную плоскость (всегда линейная для первого прохода)
    X_flat = X.flatten()
    Y_flat = Y.flatten()
    Z_flat = image_data.flatten()
    A = np.column_stack([X_flat, Y_flat, np.ones_like(X_flat)])
    coeffs, _, _, _ = np.linalg.lstsq(A, Z_flat, rcond=None)
    plane = coeffs[0] * X + coeffs[1] * Y + coeffs[2]
    temp_leveled = image_data - plane
"""
    h, w = image_data.shape
    temp_leveled = image_data
    #plt.imshow(temp_leveled)
    #plt.show()
    # Шаг 2: строим маску на временном изображении
    #print("Шаг 2/3: Построение маски островков...")
    data_norm = (image_data - np.min(image_data)) / (np.max(image_data) - np.min(image_data))
    data_norm = (data_norm * 255).astype(np.uint8)

    threshold_norm = threshold_otsu(data_norm) / 255.0
    threshold = np.min(data) + threshold_norm * (np.max(data) - np.min(data))

    mask = temp_leveled > threshold
    #plt.imshow(mask)
    #plt.show()
    
    # Расширяем маску для захвата краев
    if mask_dilation > 0:
        struct = np.ones((mask_dilation, mask_dilation))
        mask = binary_dilation(mask, structure=struct)
    
    masked_percent = np.sum(mask) * 100 / mask.size
    #print(f"  Маскировано {masked_percent:.1f}% пикселей")
    
    # Шаг 3: финальное построчное выравнивание с маской
    #print(f"Шаг 3/3: Финальное выравнивание (fit_type={fit_type}, order={order})...")
    leveled = np.zeros_like(image_data)
    x = np.arange(w)
    
    # Определяем порядок полинома
    if fit_type == 'linear':
        poly_order = 1
    elif fit_type == 'quadratic':
        poly_order = 2
    elif fit_type == 'cubic':
        poly_order = 3
    elif fit_type == 'poly':
        poly_order = order
    elif fit_type == 'mean':
        poly_order = 0
    else:
        raise ValueError(f"Неизвестный fit_type: {fit_type}. Доступны: 'poly', 'linear', 'quadratic', 'cubic', 'mean'")
    
    for i in range(h):
        line = image_data[i, :]
        line_mask = mask[i, :]
        
        # Если вся строка замаскирована, используем все точки
        if np.all(line_mask):
            x_clean = x
            y_clean = line
        else:
            x_clean = x[~line_mask]
            y_clean = line[~line_mask]
        
        # Проверяем, достаточно ли точек для аппроксимации
        min_points = poly_order + 1
        if len(x_clean) < min_points:
            # Недостаточно точек - используем все и уменьшаем порядок
            x_clean = x
            y_clean = line
            actual_order = min(1, len(x_clean) - 1)
            if actual_order < 0:
                actual_order = 0
        else:
            actual_order = poly_order
        
        # Аппроксимация полиномом
        if actual_order == 0:
            # Среднее значение
            fit_line = np.full(w, np.mean(y_clean))
        else:
            # Полином указанного порядка
            coeffs = np.polyfit(x_clean, y_clean, actual_order)
            fit_line = np.polyval(coeffs, x)
        
        leveled[i, :] = line - fit_line
    
    return leveled


# Настройки для рисования картинок

In [11]:
sns.set_context('paper')

bluetiful = '#0B68FE'
honey = '#ffcc33'
tart = '#f93f37'
lizard = '#d03f14'
violet = '#7f00ff'

plt.rcParams.update({
    'font.size': 12,              # Основной размер шрифта
    'axes.titlesize': 14,          # Размер заголовка
    'axes.labelsize': 14,          # Размер подписей осей
    'xtick.labelsize': 12,         # Размер подписей делений по X
    'ytick.labelsize': 12,         # Размер подписей делений по Y
    'legend.fontsize': 12,         # Размер шрифта легенды
    'figure.titlesize': 14,        # Размер заголовка фигуры
})

# Html-сводка по mdt-файлу

In [12]:
matplotlib.rcParams['axes.unicode_minus'] = False

def save_matplotlib_fig_to_base64(fig, dpi=100):
    """Сохраняет matplotlib figure в base64 строку с меньшим разрешением"""
    stream = BytesIO()
    fig.savefig(stream, format='png', dpi=dpi, bbox_inches='tight', facecolor='white')
    stream.seek(0)
    img_base64 = base64.b64encode(stream.read()).decode('utf-8')
    plt.close(fig)
    return img_base64

def create_frame_figure_compact(frame, dpi=200):
    """
    Создает очень компактную фигуру для одного фрейма
    """
    # Маленький размер фигуры для компактности
    fig = plt.figure(figsize=(5.5, 6))
    
    # GridSpec: изображение 80%, гистограмма 20%
    gs = fig.add_gridspec(2, 1, height_ratios=[0.8, 0.2], hspace=0.12)
    
    # Верхняя часть - изображение
    ax1 = fig.add_subplot(gs[0])
    
    # Изображение AFM
    vmin = np.percentile(frame.data_corrected, 0.5)
    vmax = np.percentile(frame.data_corrected, 99.5)
    
    im = ax1.imshow(frame.data_corrected, 
                    extent=[0, frame.xreal, 0, frame.yreal], 
                    cmap='afmhot',
                    vmin=vmin,
                    vmax=vmax)
    
    # Подписи осей с увеличенным шрифтом
    ax1.tick_params(labelsize=10)
    ax1.set_ylabel('мкм')
    # Цветовая шкала - маленькая
    cbar = plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04, shrink=0.7)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label('нм', fontsize=10)
    
    # Нижняя часть - гистограмма
    ax2 = fig.add_subplot(gs[1])
    
    # Компактная гистограмма
    ax2.hist(frame.data_corrected.flatten(), bins=25, alpha=0.7, 
             color='steelblue', edgecolor='black', linewidth=0.3)
    ax2.set_xlabel('Высота, нм', fontsize=10)
    ax2.set_yticks([])
    ax2.tick_params(labelsize=10)
    ax2.grid(True, alpha=0.2, axis='y')
    
    plt.tight_layout()
    return fig

def create_afm_viewer_html(input_filename, outdir='./'):
    """
    Создает HTML просмотрщик для AFM файлов в виде компактной сетки
    """
    sample, report_number = sample_name(input_filename)
    report_name = f'{outdir}{sample} AFM {report_number}_viewer.html'
    
    # Загружаем MDT файл
    print(f"Загрузка файла: {input_filename}")
    mdt_file = MDTFile()
    mdt_file.load_mdt_file(input_filename)
    
    # Получаем дату создания файла
    file_mtime = os.path.getmtime(input_filename)
    file_date = datetime.fromtimestamp(file_mtime).strftime("%Y-%m-%d %H:%M:%S")
    
    # Собираем данные для всех кадров
    frames_html = []
    
    for idx, frame in enumerate(mdt_file):
        # Пропускаем служебные фреймы
        if frame.title in ['Text Frame', 'Simple statistics', 'Histogram', 
                           '', 'Peak Distribution', '1F:Height Histogram']:
            continue
        
        print(f"Обработка кадра {idx}: {frame.title}")
        
        # Обрабатываем фрейм
        frame = process_frame(frame)
        
        # Создаем компактное изображение
        fig = create_frame_figure_compact(frame)
        img_base64 = save_matplotlib_fig_to_base64(fig, dpi=100)
        
        # Формируем HTML для этого фрейма
        frame_html = f'''
        <div class="card">
            <div class="card-header">
                <h4>{frame.title}</h4>
            </div>
            <div class="card-image">
                <img src="data:image/png;base64,{img_base64}" alt="{frame.title}">
            </div>
            <div class="card-stats">
                <div class="stat">
                    <span class="stat-label">Peak-to-peak</span>
                    <span class="stat-value">{frame.peak_to_peak:.1f} нм</span>
                </div>
                <div class="stat">
                    <span class="stat-label">RMS</span>
                    <span class="stat-value">{frame.rms:.1f} нм</span>
                </div>
                <div class="stat">
                    <span class="stat-label">RMS masked</span>
                    <span class="stat-value">{frame.rms_otsu:.1f} нм</span>
                </div>
            </div>
        </div>
        '''
        frames_html.append(frame_html)
    
    # Создаем полный HTML
    filename_only = Path(input_filename).name
    
    html_content = f'''<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AFM Viewer - {sample}</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: #e8eef2;
            padding: 15px;
        }}
        
        .container {{
            max-width: 1400px;
            margin: 0 auto;
        }}
        
        .header {{
            background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
            color: white;
            padding: 15px 20px;
            border-radius: 10px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        
        .header h1 {{
            font-size: 22px;
            margin-bottom: 10px;
        }}
        
        .header-info {{
            display: flex;
            flex-wrap: wrap;
            gap: 15px;
            font-size: 12px;
            opacity: 0.9;
        }}
        
        .header-info span {{
            color: #ffd700;
        }}
        
        /* Сетка карточек - более плотная */
        .grid {{
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(280px, 1fr));
            gap: 15px;
            margin-bottom: 20px;
        }}
        
        /* Карточка - компактная */
        .card {{
            background: white;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 1px 3px rgba(0,0,0,0.08);
            display: flex;
            flex-direction: column;
            transition: box-shadow 0.2s;
        }}
        
        .card:hover {{
            box-shadow: 0 2px 8px rgba(0,0,0,0.12);
        }}
        
        .card-header {{
            background: #f5f5f5;
            padding: 8px 10px;
            border-bottom: 1px solid #e0e0e0;
        }}
        
        .card-header h4 {{
            color: #1e3c72;
            font-size: 13px;
            font-weight: 600;
            margin: 0;
            text-align: center;
        }}
        
        .card-image {{
            background: #fafafa;
            padding: 5px;
            text-align: center;
        }}
        
        .card-image img {{
            width: 100%;
            height: auto;
            display: block;
            border-radius: 4px;
        }}
        
        .card-stats {{
            padding: 8px 10px;
            background: white;
            border-top: 1px solid #eee;
            display: flex;
            flex-direction: column;
            gap: 4px;
        }}
        
        .stat {{
            display: flex;
            justify-content: space-between;
            align-items: baseline;
            padding: 3px 0;
        }}
        
        .stat-label {{
            font-size: 10px;
            color: #555;
            font-weight: 500;
        }}
        
        .stat-value {{
            font-size: 11px;
            font-weight: 600;
            color: #2a5298;
            font-family: 'Courier New', monospace;
        }}
        
        .footer {{
            text-align: center;
            padding: 15px;
            color: #6c757d;
            font-size: 10px;
            margin-top: 15px;
            border-top: 1px solid #dee2e6;
        }}
        
        /* Медиа-запросы */
        @media (max-width: 1000px) {{
            .grid {{
                grid-template-columns: repeat(auto-fill, minmax(260px, 1fr));
                gap: 12px;
            }}
        }}
        
        @media (max-width: 768px) {{
            body {{
                padding: 10px;
            }}
            .grid {{
                grid-template-columns: repeat(auto-fill, minmax(240px, 1fr));
                gap: 10px;
            }}
            .header-info {{
                flex-direction: column;
                gap: 5px;
            }}
        }}
        
        /* Для печати */
        @media print {{
            body {{
                background: white;
                padding: 0;
            }}
            .card {{
                break-inside: avoid;
                page-break-inside: avoid;
                box-shadow: none;
                border: 1px solid #ddd;
            }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🔬 {sample}</h1>
            <div class="header-info">
                <div>📁 <span>{filename_only}</span></div>
                <div>📅 <span>{file_date}</span></div>
                <div>📊 <span>{len(frames_html)} кадров</span></div>
            </div>
        </div>
        
        <div class="grid">
            {''.join(frames_html)}
        </div>
        
        <div class="footer">
            <p>🔬 Создано автоматически | AFM Viewer</p>
        </div>
    </div>
</body>
</html>
'''
    
    # Сохраняем HTML файл
    with open(report_name, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"\n✅ Отчет сохранен: {report_name}")
    print(f"📊 Обработано кадров: {len(frames_html)}")
    
    return report_name